# 第10课：PCA 降维

## 学习目标
- 理解「降维」的直觉：为什么要压缩特征？
- 掌握 PCA 的核心原理：方差最大化 + 正交投影
- 从零实现 PCA，理解协方差矩阵和特征分解
- 学会用 PCA 做数据可视化、噪声过滤和特征压缩

## 核心概念

上一课 KMeans 是无监督学习中的「聚类」分支——没有标签，找分组。

今天进入无监督学习的另一个分支：**降维（Dimensionality Reduction）**。

**直觉**：想象你在拍一张 3D 物体的照片。你从某个角度拍，能最好地保留物体的「形状信息」。PCA 就是自动找到最佳拍摄角度的算法——把高维数据投影到低维空间，同时尽量保留信息。

**为什么需要降维？**
1. **维度灾难**：特征太多 → 样本稀疏 → 模型难学
2. **可视化**：人只能看 2D/3D，高维数据看不到分布
3. **去噪**：小方差维度往往是噪声
4. **加速**：特征少了，训练更快

**PCA 的核心思路**：
1. 数据中心化（减去均值）
2. 计算协方差矩阵
3. 对协方差矩阵做特征分解，取最大特征值对应的特征向量
4. 用这些特征向量作为新的「坐标轴」

**关键洞察**：PCA 找的是「方差最大的方向」——因为方差大 = 信息多。第一个主成分是方差最大的方向，第二个是与之正交且方差次大的方向，以此类推。

**在 AI 演进中的位置**：
- 1901：Pearson 提出主成分分析
- 1933：Hotelling 独立发展并命名 PCA
- 至今仍是数据分析、特征工程、可视化的标准工具
- 深度学习中的自编码器可以看作非线性版的 PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.font_manager as fm
import pathlib
_cache = matplotlib.get_cachedir()
for _f in pathlib.Path(_cache).glob('fontlist*.json'):
    _f.unlink(missing_ok=True)
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.sans-serif'] = ['PingFang HK', 'PingFang SC', 'Heiti TC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.decomposition import PCA as SklearnPCA
from sklearn.datasets import load_iris, load_digits
from sklearn.preprocessing import StandardScaler

# 设置中文显示
np.random.seed(42)

## 1. 从零实现 PCA

PCA 的数学本质就是 4 步：
1. **中心化**：X_centered = X - mean(X)
2. **协方差矩阵**：C = X_centered.T @ X_centered / (n-1)
3. **特征分解**：对 C 做特征分解，得到特征值和特征向量
4. **投影**：用前 k 个特征向量（按特征值从大到小排列）做投影

In [ ]:
class MyPCA:
    """从零实现的 PCA"""
    
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.components_ = None  # 主成分方向
        self.explained_variance_ = None  # 各主成分的方差
        self.explained_variance_ratio_ = None  # 方差占比
        self.mean_ = None
    
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        
        # 协方差矩阵
        n = X_centered.shape[0]
        cov_matrix = X_centered.T @ X_centered / (n - 1)
        
        # 特征分解
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # 按特征值从大到小排序
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # 取前 k 个
        self.components_ = eigenvectors[:, :self.n_components].T
        self.explained_variance_ = eigenvalues[:self.n_components]
        total_var = eigenvalues.sum()
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / total_var
        return self
    
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# 用鸢尾花数据集验证
iris = load_iris()
X = iris.data
y = iris.target

my_pca = MyPCA(n_components=2)
X_my = my_pca.fit_transform(X)

sk_pca = SklearnPCA(n_components=2)
X_sk = sk_pca.fit_transform(X)

print(f"自定义PCA方差占比: {my_pca.explained_variance_ratio_}")
print(f"sklearn PCA方差占比: {sk_pca.explained_variance_ratio_}")
print(f"结果接近: {np.allclose(np.abs(X_my), np.abs(X_sk), atol=0.01)}")

## 2. 可视化：4D 数据降到 2D 看分布

鸢尾花有 4 个特征（花萼长宽、花瓣长宽），人眼看不到 4D 分布。PCA 降到 2D 后，三类花的分布一目了然。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']
target_names = iris.target_names

# 自定义 PCA
for i, name in enumerate(target_names):
    mask = y == i
    axes[0].scatter(X_my[mask, 0], X_my[mask, 1], c=colors[i], label=name, alpha=0.7, s=50)
axes[0].set_title('MyPCA - Iris 降到 2D')
axes[0].set_xlabel(f'PC1 ({my_pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({my_pca.explained_variance_ratio_[1]:.1%})')
axes[0].legend()

# sklearn PCA
for i, name in enumerate(target_names):
    mask = y == i
    axes[1].scatter(X_sk[mask, 0], X_sk[mask, 1], c=colors[i], label=name, alpha=0.7, s=50)
axes[1].set_title('sklearn PCA - Iris 降到 2D')
axes[1].set_xlabel(f'PC1 ({sk_pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({sk_pca.explained_variance_ratio_[1]:.1%})')
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_iris_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"前两个主成分保留了 {sum(my_pca.explained_variance_ratio_):.1%} 的信息")

## 3. 手写数字降维 + 方差解释曲线

MNIST 手写数字有 64 个特征（8×8 像素）。我们用 PCA 看看：
- 前几个主成分能保留多少信息？
- 降到 2D 后，0-9 的数字是否自然分开？

In [ ]:
# 手写数字数据集
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# 先标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_digits)

pca_digits = SklearnPCA(n_components=30)
X_pca = pca_digits.fit_transform(X_scaled)

# 方差解释曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 累积方差
cumvar = np.cumsum(pca_digits.explained_variance_ratio_)
axes[0].plot(range(1, 31), cumvar, 'o-', color='#C96442', markersize=4)
axes[0].axhline(y=0.9, color='gray', linestyle='--', label='90%')
axes[0].axhline(y=0.95, color='gray', linestyle=':', label='95%')
axes[0].set_xlabel('主成分数量')
axes[0].set_ylabel('累积方差解释比例')
axes[0].set_title('PCA 累积方差解释曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 找到 90% 和 95% 需要几个主成分
n90 = np.argmax(cumvar >= 0.9) + 1
n95 = np.argmax(cumvar >= 0.95) + 1
print(f"90% 方差需要 {n90} 个主成分")
print(f"95% 方差需要 {n95} 个主成分")
print(f"原始维度: {X_digits.shape[1]}")

# 2D 可视化
pca2d = SklearnPCA(n_components=2)
X_2d = pca2d.fit_transform(X_scaled)
scatter = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.5, s=15)
axes[1].set_title(f'手写数字 PCA 2D (保留 {sum(pca2d.explained_variance_ratio_):.1%} 方差)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[1], label='数字')

plt.tight_layout()
plt.savefig('pca_digits.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. PCA 去噪实验

PCA 的一个巧妙应用：**去噪**。思路是只保留大的主成分（信号），丢弃小的（噪声）。

In [ ]:
# 给手写数字加噪声，然后用 PCA 去噪
np.random.seed(42)
noise = np.random.normal(0, 3, X_digits.shape)
X_noisy = X_digits + noise

fig, axes = plt.subplots(3, 5, figsize=(12, 7))

for col in range(5):
    img_idx = col * 3
    # 原始
    axes[0, col].imshow(X_digits[img_idx].reshape(8, 8), cmap='gray')
    axes[0, col].set_title(f'原始 #{img_idx}')
    axes[0, col].axis('off')
    
    # 加噪
    axes[1, col].imshow(X_noisy[img_idx].reshape(8, 8), cmap='gray')
    axes[1, col].set_title('加噪')
    axes[1, col].axis('off')
    
    # PCA 去噪
    pca_denoise = SklearnPCA(n_components=20)
    X_reconstructed = pca_denoise.fit_transform(X_noisy).dot(pca_denoise.components_) + pca_denoise.mean_
    axes[2, col].imshow(X_reconstructed[img_idx].reshape(8, 8), cmap='gray')
    axes[2, col].set_title(f'PCA去噪(k=20)')
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('原始', fontsize=12)
axes[1, 0].set_ylabel('加噪', fontsize=12)
axes[2, 0].set_ylabel('PCA去噪', fontsize=12)

plt.suptitle('PCA 去噪效果', fontsize=14)
plt.tight_layout()
plt.savefig('pca_denoise.png', dpi=100, bbox_inches='tight')
plt.show()

# 计算重建误差
from sklearn.metrics import mean_squared_error
mse_noisy = mean_squared_error(X_digits, X_noisy)
mse_denoised = mean_squared_error(X_digits, X_reconstructed)
print(f"噪声图像 MSE: {mse_noisy:.2f}")
print(f"去噪后 MSE:   {mse_denoised:.2f}")
print(f"误差降低:     {(1 - mse_denoised/mse_noisy):.1%}")

## 关键要点

| 概念 | 说明 |
|------|------|
| 协方差矩阵 | 描述特征之间的线性相关性，是 PCA 的输入 |
| 特征值/特征向量 | 特征值 = 方差大小，特征向量 = 投影方向 |
| 主成分 | 按方差从大到小排列的新坐标轴 |
| explained_variance_ratio | 每个主成分保留了多少信息 |

**记住这三个用法**：
1. **可视化**：高维 → 2D/3D 看分布
2. **压缩**：减少特征数，加速训练
3. **去噪**：丢弃小方差维度，保留信号

**下一步**：模型评估与调参实战——用 GridSearchCV 系统调参，为进入深度学习做最后准备。